# 02 — Data Preparation

This notebook materializes a deterministic, auditable preparation handoff for the Telco Customer Churn study. It is intentionally self-contained and must run from a fresh kernel without variables from Notebook 01.


## 1. Preparation Context

Notebook 02 owns deterministic data correction, role separation, educational snapshot partitioning, local persistence, and the handoff to model selection.

The approved evaluation mode is `stratified_random_snapshot` for an `educational_benchmark`. This decision permits educational model selection in Notebook 03, but it does **not** establish temporal validity, future-customer performance, feature availability at inference time, or production readiness. Learned preprocessing, feature selection, resampling, model fitting, threshold selection, and evaluation remain outside this notebook.


In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

from scripts.download_data import DatasetDownloadError, acquire_kaggle_dataset
from scripts.prepare_data import (
    ClassificationSplitPolicy,
    ConditionalNumericRule,
    build_feature_manifest,
    build_preparation_manifest,
    build_quality_evidence,
    build_split_manifest,
    fingerprint_dataframe,
    fingerprint_dataframe_csv,
    fingerprint_file,
    load_and_validate_preparation_handoff,
    prepare_tabular_dataset,
    separate_dataset_roles,
    split_classification_dataset,
    validate_dataset_partitions,
    validate_prepared_dataset,
    validate_raw_dataset,
    write_preparation_artifacts,
)
from scripts.project_context import get_project_context

PROJECT = get_project_context()

DATASET_SLUG = "telco-customer-churn"
DATASET_HANDLE = "blastchar/telco-customer-churn"
DATASET_FILE_SELECTOR = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
CSV_READ_OPTIONS = {}

TARGET_COLUMN = "Churn"
IDENTIFIER_COLUMNS = ("customerID",)
TARGET_CLASSES = ("No", "Yes")
POSITIVE_TARGET_CLASS = "Yes"
TARGET_ENCODING = {"No": 0, "Yes": 1}

NUMERICAL_FEATURES = (
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
)

CATEGORICAL_FEATURES = (
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
)

FEATURE_COLUMNS = (
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
)

COLUMN_ORDER = (
    "customerID",
    *FEATURE_COLUMNS,
    "Churn",
)

CATEGORICAL_EXPECTED_VALUES = {
    "gender": ("Female", "Male"),
    "SeniorCitizen": (0, 1),
    "Partner": ("No", "Yes"),
    "Dependents": ("No", "Yes"),
    "PhoneService": ("No", "Yes"),
    "MultipleLines": ("No", "Yes", "No phone service"),
    "InternetService": ("DSL", "Fiber optic", "No"),
    "OnlineSecurity": ("No", "Yes", "No internet service"),
    "OnlineBackup": ("No", "Yes", "No internet service"),
    "DeviceProtection": ("No", "Yes", "No internet service"),
    "TechSupport": ("No", "Yes", "No internet service"),
    "StreamingTV": ("No", "Yes", "No internet service"),
    "StreamingMovies": ("No", "Yes", "No internet service"),
    "Contract": ("Month-to-month", "One year", "Two year"),
    "PaperlessBilling": ("No", "Yes"),
    "PaymentMethod": (
        "Electronic check",
        "Mailed check",
        "Bank transfer (automatic)",
        "Credit card (automatic)",
    ),
}

RAW_EXPECTED_DATA_TYPES = {
    "customerID": "string",
    "gender": "string",
    "SeniorCitizen": "integer",
    "Partner": "string",
    "Dependents": "string",
    "tenure": "integer",
    "PhoneService": "string",
    "MultipleLines": "string",
    "InternetService": "string",
    "OnlineSecurity": "string",
    "OnlineBackup": "string",
    "DeviceProtection": "string",
    "TechSupport": "string",
    "StreamingTV": "string",
    "StreamingMovies": "string",
    "Contract": "string",
    "PaperlessBilling": "string",
    "PaymentMethod": "string",
    "MonthlyCharges": "numeric",
    "TotalCharges": "numeric",
    "Churn": "string",
}
PREPARED_EXPECTED_DATA_TYPES = dict(RAW_EXPECTED_DATA_TYPES)

EXPECTED_SOURCE_ROWS = 7_043
EXPECTED_SOURCE_COLUMNS = 21
EXPECTED_TOTAL_CHARGES_MATERIALIZATIONS = 11

print("Preparation context initialized for:", DATASET_SLUG)
print("Project:", PROJECT.name)


Preparation context initialized for: telco-customer-churn
Project: dataset-study-telco-customer-churn


## 2. Independent Project and Dataset Loading

The raw source is reused when already present. Otherwise, the existing Kaggle acquisition helper is invoked. Selection is exact and deterministic; the notebook never depends on a DataFrame or path left alive by Notebook 01.


In [2]:
RAW_DATA_DIR = PROJECT.path("data", "raw", DATASET_SLUG)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


def select_source_csv() -> Path:
    matches = tuple(
        sorted(
            (
                path.resolve()
                for path in RAW_DATA_DIR.rglob(DATASET_FILE_SELECTOR)
                if path.is_file()
            ),
            key=lambda path: path.as_posix(),
        )
    )
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            "Deterministic source selection failed: more than one file "
            f"matches {DATASET_FILE_SELECTOR!r}."
        )

    try:
        acquire_kaggle_dataset(
            handle=DATASET_HANDLE,
            destination=Path("data") / "raw" / DATASET_SLUG,
            dataset_file=DATASET_FILE_SELECTOR,
            force=False,
            show_progress=False,
            project_root=PROJECT.root,
        )
    except DatasetDownloadError as exc:
        raise RuntimeError(
            "The raw dataset is not available locally and acquisition failed. "
            "Restore data/raw/telco-customer-churn or provide network access "
            "to the public Kaggle source before executing this notebook."
        ) from exc

    matches = tuple(
        sorted(
            (
                path.resolve()
                for path in RAW_DATA_DIR.rglob(DATASET_FILE_SELECTOR)
                if path.is_file()
            ),
            key=lambda path: path.as_posix(),
        )
    )
    if len(matches) != 1:
        raise RuntimeError(
            "Acquisition completed but deterministic CSV selection did not "
            f"resolve exactly one {DATASET_FILE_SELECTOR!r}."
        )
    return matches[0]


DATASET_FILE = select_source_csv()
raw_df = pd.read_csv(DATASET_FILE, **CSV_READ_OPTIONS)

print("Source:", PROJECT.display(DATASET_FILE))
print("Shape:", raw_df.shape)


Source: data/raw/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


## 3. Source Dataset Fingerprint

Source bytes and the logical DataFrame projection are fingerprinted before any transformation. The snapshot below is used later to prove raw immutability, row preservation, identifier preservation, and source-file preservation.


In [3]:
SOURCE_RELATIVE_PATH = PROJECT.display(DATASET_FILE)
SOURCE_SHA256_BEFORE = fingerprint_file(DATASET_FILE)
RAW_LOGICAL_FINGERPRINT_BEFORE = fingerprint_dataframe(raw_df)
RAW_SNAPSHOT = raw_df.copy(deep=True)
RAW_INDEX_SNAPSHOT = raw_df.index.copy(deep=True)
RAW_CUSTOMER_IDS_SNAPSHOT = raw_df.loc[:, list(IDENTIFIER_COLUMNS)].copy(deep=True)
RAW_COLUMN_ORDER_SNAPSHOT = tuple(raw_df.columns)
RAW_DTYPES_SNAPSHOT = tuple((column, str(raw_df[column].dtype)) for column in raw_df.columns)

source_summary = pd.DataFrame(
    [
        {
            "source_path": SOURCE_RELATIVE_PATH,
            "source_sha256": SOURCE_SHA256_BEFORE,
            "logical_fingerprint": RAW_LOGICAL_FINGERPRINT_BEFORE,
            "rows": len(raw_df),
            "columns": len(raw_df.columns),
        }
    ]
)
display(source_summary)


,source_path,source_sha256,logical_fingerprint,rows,columns
0,data/raw/telco-customer-churn/WA_Fn-UseC_-Telc...,88be4b93fbe0cc83421af1c503794c97c342eca914c157...,90d72b056df6f126b3341cb782928a8624801eece8ac45...,7043,21


## 4. Raw Schema and Role Validation

The raw contract validates required fields, exact column order, unique non-blank identifiers, target completeness, expected classes, declared categorical domains, semantic types, and the controlled numeric-text exception for `TotalCharges`.


In [4]:
raw_validation = validate_raw_dataset(
    raw_df,
    column_order=COLUMN_ORDER,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    categorical_expected_values=CATEGORICAL_EXPECTED_VALUES,
    expected_types=RAW_EXPECTED_DATA_TYPES,
    numeric_text_columns=("TotalCharges",),
)

if raw_validation.row_count != EXPECTED_SOURCE_ROWS:
    raise RuntimeError(
        f"Unexpected Telco source row count: {raw_validation.row_count}; "
        f"expected {EXPECTED_SOURCE_ROWS}."
    )
if raw_validation.column_count != EXPECTED_SOURCE_COLUMNS:
    raise RuntimeError(
        f"Unexpected Telco source column count: {raw_validation.column_count}; "
        f"expected {EXPECTED_SOURCE_COLUMNS}."
    )

print("Raw schema valid:", raw_validation.is_valid)
print("Identifier columns:", IDENTIFIER_COLUMNS)
print("Predictor count:", len(FEATURE_COLUMNS))
print("Target classes:", TARGET_CLASSES)


Raw schema valid: True
Identifier columns: ('customerID',)
Predictor count: 19
Target classes: ('No', 'Yes')


## 5. Defensive Prepared Projection

Preparation starts from a deep copy. The raw DataFrame, raw index, source identifiers, source columns, and source dtypes remain independently tracked.


In [5]:
TOTAL_CHARGES_RULE = ConditionalNumericRule(
    column="TotalCharges",
    condition_column="tenure",
    condition_value=0,
    blank_replacement=0.0,
    strip_strings=True,
)

prepared_result = prepare_tabular_dataset(
    raw_df,
    conditional_numeric_rules=(TOTAL_CHARGES_RULE,),
)
prepared_df = prepared_result.dataframe

if prepared_df is raw_df:
    raise RuntimeError("Prepared projection unexpectedly aliases raw_df.")

print("Defensive prepared projection created.")
print("Raw dtype:", raw_df["TotalCharges"].dtype)
print("Prepared dtype:", prepared_df["TotalCharges"].dtype)


Defensive prepared projection created.
Raw dtype: str
Prepared dtype: float64


## 6. Deterministic TotalCharges Materialization

The only authorized value correction strips textual whitespace, converts blank strings to `0.0` only when `tenure == 0`, rejects blanks with positive tenure, rejects any remaining non-numeric value, and converts `TotalCharges` to a numeric dtype. No statistical imputation, row removal, clipping, outlier treatment, or category rewriting is performed.


In [6]:
TOTAL_CHARGES_MATERIALIZED = prepared_result.materialization_count("TotalCharges")
TOTAL_CHARGES_INVALID = dict(prepared_result.invalid_conversion_counts)["TotalCharges"]

if TOTAL_CHARGES_MATERIALIZED != EXPECTED_TOTAL_CHARGES_MATERIALIZATIONS:
    raise RuntimeError(
        "Unexpected TotalCharges materialization count: "
        f"{TOTAL_CHARGES_MATERIALIZED}; expected "
        f"{EXPECTED_TOTAL_CHARGES_MATERIALIZATIONS}."
    )
if TOTAL_CHARGES_INVALID != 0:
    raise RuntimeError("Invalid TotalCharges conversions remain after preparation.")
if prepared_df["TotalCharges"].isna().any():
    raise RuntimeError("Missing TotalCharges values remain after preparation.")

print("TotalCharges values materialized:", TOTAL_CHARGES_MATERIALIZED)
print("Invalid conversions remaining:", TOTAL_CHARGES_INVALID)
print("Prepared dtype:", prepared_df["TotalCharges"].dtype)


TotalCharges values materialized: 11
Invalid conversions remaining: 0
Prepared dtype: float64


## 7. Post-Preparation Quality Validation

The post-preparation contract requires the original 7,043 rows and 21 columns, unchanged identifier/target/category values, unchanged order, no unauthorized mutation, and no invalid or missing `TotalCharges` values.


In [7]:
prepared_validation = validate_prepared_dataset(
    raw_df,
    prepared_df,
    column_order=COLUMN_ORDER,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    categorical_expected_values=CATEGORICAL_EXPECTED_VALUES,
    expected_types=PREPARED_EXPECTED_DATA_TYPES,
    authorized_changed_columns=("TotalCharges",),
    expected_row_count=EXPECTED_SOURCE_ROWS,
    expected_materialized_counts={
        "TotalCharges": EXPECTED_TOTAL_CHARGES_MATERIALIZATIONS,
    },
    observed_materialized_counts=dict(prepared_result.materialized_counts),
)

pd.testing.assert_frame_equal(raw_df, RAW_SNAPSHOT)
if not raw_df.index.equals(RAW_INDEX_SNAPSHOT):
    raise RuntimeError("raw_df index changed during preparation.")
pd.testing.assert_frame_equal(
    raw_df.loc[:, list(IDENTIFIER_COLUMNS)],
    RAW_CUSTOMER_IDS_SNAPSHOT,
)
if tuple(raw_df.columns) != RAW_COLUMN_ORDER_SNAPSHOT:
    raise RuntimeError("raw_df column order changed during preparation.")
if tuple((column, str(raw_df[column].dtype)) for column in raw_df.columns) != RAW_DTYPES_SNAPSHOT:
    raise RuntimeError("raw_df dtypes changed during preparation.")

SOURCE_SHA256_AFTER_PREPARATION = fingerprint_file(DATASET_FILE)
RAW_LOGICAL_FINGERPRINT_AFTER = fingerprint_dataframe(raw_df)

if SOURCE_SHA256_AFTER_PREPARATION != SOURCE_SHA256_BEFORE:
    raise RuntimeError("The source CSV bytes changed during preparation.")
if RAW_LOGICAL_FINGERPRINT_AFTER != RAW_LOGICAL_FINGERPRINT_BEFORE:
    raise RuntimeError("The raw DataFrame logical fingerprint changed.")

print("Prepared dataset valid:", prepared_validation.is_valid)
print("Rows removed: 0")
print("Identifier values changed: 0")
print("Raw source unchanged: True")


Prepared dataset valid: True
Rows removed: 0
Identifier values changed: 0
Raw source unchanged: True


## 8. Identifier, Feature, and Target Separation

`customerID` remains lineage-only, the predictor matrix contains exactly 19 declared features in source order, and `Churn` remains readable as `No`/`Yes`. The numeric target mapping is a future training-interface contract rather than a replacement for the readable persisted target.


In [8]:
roles = separate_dataset_roles(
    prepared_df,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
)
lineage = roles.lineage
X = roles.features
y = roles.target

if tuple(X.columns) != FEATURE_COLUMNS:
    raise RuntimeError("Predictor order differs from FEATURE_COLUMNS.")
if any(column in X.columns for column in (*IDENTIFIER_COLUMNS, TARGET_COLUMN)):
    raise RuntimeError("Identifier or target leaked into X.")
if tuple(pd.unique(y)) and not set(pd.unique(y)).issubset(TARGET_CLASSES):
    raise RuntimeError("Readable target contains unexpected values.")

print("Lineage columns:", tuple(lineage.columns))
print("X shape:", X.shape)
print("y classes:", tuple(sorted(pd.unique(y))))
print("Future target encoding:", TARGET_ENCODING)


Lineage columns: ('customerID',)
X shape: (7043, 19)
y classes: ('No', 'Yes')
Future target encoding: {'No': 0, 'Yes': 1}


## 9. Split-Policy Declaration

The benchmark uses a deterministic two-stage stratified random split. The temporary set is separated first, then divided into validation and test. Scikit-learn float `test_size` semantics round each held-out size upward with `ceil`; the remainder is assigned to the first set. Seeds are recorded explicitly.


In [9]:
SPLIT_POLICY = ClassificationSplitPolicy(
    evaluation_mode="stratified_random_snapshot",
    purpose="educational_benchmark",
    train_fraction=0.70,
    validation_fraction=0.15,
    test_fraction=0.15,
    stratify_by=TARGET_COLUMN,
    random_seed=42,
    shuffle=True,
    educational_justification=(
        "Advance reproducible educational model selection while explicitly "
        "withholding claims about temporal, future-customer, inference-time, "
        "or production validity."
    ),
    operational_validity="unconfirmed",
    temporal_contract_status="unresolved",
    feature_inference_availability="unconfirmed",
)

split_policy_summary = pd.DataFrame([SPLIT_POLICY.as_dict()]).drop(columns=["stage_seeds"])
display(split_policy_summary)
print("Stage seeds:", SPLIT_POLICY.as_dict()["stage_seeds"])


,evaluation_mode,purpose,train_fraction,validation_fraction,test_fraction,stratify_by,random_seed,shuffle,educational_justification,operational_validity,temporal_contract_status,feature_inference_availability
0,stratified_random_snapshot,educational_benchmark,0.7,0.15,0.15,Churn,42,True,Advance reproducible educational model selecti...,unconfirmed,unresolved,unconfirmed


Stage seeds: {'train_vs_temporary': 42, 'validation_vs_test': 43}


## 10. Dataset Partitioning

Membership is generated after stable identifier ordering, which makes it independent of incidental input row order. Rows inside each returned partition are restored to original source position. No encoder, scaler, selector, resampler, interaction, model, threshold, or learned transformation is fitted here.


In [10]:
partitions = split_classification_dataset(
    prepared_df,
    policy=SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=TARGET_CLASSES,
)

train_df = partitions.train
validation_df = partitions.validation
test_df = partitions.test

partition_sizes = pd.DataFrame(
    [
        {"partition": "train", "rows": len(train_df)},
        {"partition": "validation", "rows": len(validation_df)},
        {"partition": "test", "rows": len(test_df)},
    ]
)
display(partition_sizes)
print("Split method:", partitions.split_method)
print("Rounding:", partitions.rounding_method)


,partition,rows
0,train,4930
1,validation,1056
2,test,1057


Split method: two_stage_sklearn_train_test_split_with_stratification_and_identifier_sorted_membership
Rounding: scikit-learn float test_size semantics: each held-out size is rounded up with ceil; the remainder is assigned to the first set


## 11. Partition Integrity Validation

All partitions must contain both classes, preserve prevalence within a documented tolerance, remain disjoint by customer membership, cover every source row exactly once, preserve stable source order, and reproduce identical membership with the same policy and seeds. The test set is a strict holdout and is not used for any decision.


In [11]:
PARTITION_PREVALENCE_TOLERANCE = 0.01
partition_validation = validate_dataset_partitions(
    prepared_df,
    partitions,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    prevalence_tolerance=PARTITION_PREVALENCE_TOLERANCE,
)

repeat_partitions = split_classification_dataset(
    prepared_df,
    policy=SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=TARGET_CLASSES,
)
for partition_name in ("train", "validation", "test"):
    first_ids = list(partitions.as_mapping()[partition_name]["customerID"])
    repeat_ids = list(repeat_partitions.as_mapping()[partition_name]["customerID"])
    if first_ids != repeat_ids:
        raise RuntimeError(f"Split reproducibility failed for {partition_name}.")

reordered_df = prepared_df.sample(frac=1.0, random_state=999)
reordered_partitions = split_classification_dataset(
    reordered_df,
    policy=SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=TARGET_CLASSES,
)
for partition_name in ("train", "validation", "test"):
    expected_membership = set(partitions.as_mapping()[partition_name]["customerID"])
    observed_membership = set(reordered_partitions.as_mapping()[partition_name]["customerID"])
    if expected_membership != observed_membership:
        raise RuntimeError(
            f"Identifier-stable membership failed for {partition_name}."
        )

prevalence_rows = []
for name, frame in partitions.as_mapping().items():
    counts = frame[TARGET_COLUMN].value_counts().reindex(TARGET_CLASSES, fill_value=0)
    for target_class in TARGET_CLASSES:
        prevalence_rows.append(
            {
                "partition": name,
                "class": target_class,
                "count": int(counts[target_class]),
                "prevalence": float((frame[TARGET_COLUMN] == target_class).mean()),
            }
        )
display(pd.DataFrame(prevalence_rows))
print("Partition integrity valid:", partition_validation.is_valid)
print("Test holdout isolated: True")


,partition,class,count,prevalence
0,train,No,3622,0.734686
1,train,Yes,1308,0.265314
2,validation,No,776,0.734848
3,validation,Yes,280,0.265152
4,test,No,776,0.734153
5,test,Yes,281,0.265847


Partition integrity valid: True
Test holdout isolated: True


## 12. Preprocessing Contract for Model Selection

Notebook 03 must fit category encoding, scaling, selection, and any other learned transformation **inside each training fold**. Unknown categories are ignored by the encoder but must be reported. Numerical scaling remains model-specific. No definitive one-hot matrix, scaler, imputer, selector, resampling strategy, class weighting, interaction, logarithmic transform, threshold, model, or hyperparameter is produced here.


In [12]:
PREPROCESSING_CONTRACT = {
    "categorical_strategy": "one_hot",
    "categorical_fit_scope": "inside_training_fold",
    "unknown_category_policy": "ignore_and_report",
    "drop_category": None,
    "numerical_scaling": "model_specific",
    "numerical_fit_scope": "inside_training_fold",
    "learned_transformations_fitted_in_notebook_02": False,
    "deferred_operations": [
        "TotalCharges log1p comparison",
        "scaler comparison",
        "tenure and TotalCharges ablation",
        "regularization",
        "feature interactions",
        "class weights",
        "oversampling",
        "undersampling",
        "SMOTE",
        "feature selection",
        "model comparison",
        "threshold selection",
    ],
}

display(pd.DataFrame([PREPROCESSING_CONTRACT]).drop(columns=["deferred_operations"]))
print("Deferred operation count:", len(PREPROCESSING_CONTRACT["deferred_operations"]))


,categorical_strategy,categorical_fit_scope,unknown_category_policy,drop_category,numerical_scaling,numerical_fit_scope,learned_transformations_fitted_in_notebook_02
0,one_hot,inside_training_fold,ignore_and_report,None,model_specific,inside_training_fold,False


Deferred operation count: 12


## 13. Prepared Data and Artifact Materialization

Prepared data and split files are local, reproducible, non-versioned runtime outputs. CSV bytes are serialized deterministically before manifest construction so every SHA-256 records the bytes that will actually be written. The writer stages and validates the complete set, rejects semantic conflicts by default, accepts equivalent reruns, and promotes with rollback protection.


In [13]:
PREPARED_RELATIVE_PATH = (
    Path("data") / "processed" / DATASET_SLUG / "prepared.csv"
)
SPLIT_RELATIVE_DIR = (
    Path("data")
    / "processed"
    / DATASET_SLUG
    / "splits"
    / "stratified-random-snapshot-seed-42"
)
PARTITION_RELATIVE_PATHS = {
    "train": SPLIT_RELATIVE_DIR / "train.csv",
    "validation": SPLIT_RELATIVE_DIR / "validation.csv",
    "test": SPLIT_RELATIVE_DIR / "test.csv",
}
ARTIFACT_RELATIVE_DIR = Path("artifacts") / "preparation" / DATASET_SLUG
MANIFEST_RELATIVE_PATHS = {
    "preparation_manifest": ARTIFACT_RELATIVE_DIR / "preparation-manifest.json",
    "feature_manifest": ARTIFACT_RELATIVE_DIR / "feature-manifest.json",
    "split_manifest": ARTIFACT_RELATIVE_DIR / "split-manifest.json",
    "quality_evidence": ARTIFACT_RELATIVE_DIR / "quality-evidence.json",
}

PREPARED_SHA256 = fingerprint_dataframe_csv(prepared_df)
PARTITION_SHA256 = {
    name: fingerprint_dataframe_csv(frame)
    for name, frame in partitions.as_mapping().items()
}

READINESS_STATES = {
    "deterministic_preparation_ready": True,
    "prepared_dataset_materialized": True,
    "benchmark_split_ready": True,
    "benchmark_partitions_materialized": True,
    "educational_model_selection_ready": True,
    "operational_modeling_ready": False,
    "temporal_contract_status": "unresolved",
    "feature_inference_availability": "unconfirmed",
    "operational_validity": "unconfirmed",
}

preparation_manifest = build_preparation_manifest(
    dataset_slug=DATASET_SLUG,
    source_path=SOURCE_RELATIVE_PATH,
    source_sha256=SOURCE_SHA256_BEFORE,
    prepared_path=PREPARED_RELATIVE_PATH,
    prepared_sha256=PREPARED_SHA256,
    raw_report=raw_validation,
    prepared_report=prepared_validation,
    preparation=prepared_result,
    raw_fingerprint_before=RAW_LOGICAL_FINGERPRINT_BEFORE,
    raw_fingerprint_after=RAW_LOGICAL_FINGERPRINT_AFTER,
    source_sha256_after=SOURCE_SHA256_AFTER_PREPARATION,
    deterministic_rules=[TOTAL_CHARGES_RULE.as_dict()],
    readiness=READINESS_STATES,
)

feature_manifest = build_feature_manifest(
    dataset_slug=DATASET_SLUG,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    categorical_expected_values=CATEGORICAL_EXPECTED_VALUES,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    positive_target_class=POSITIVE_TARGET_CLASS,
    target_encoding=TARGET_ENCODING,
    expected_dtypes={
        "raw": RAW_EXPECTED_DATA_TYPES,
        "prepared": PREPARED_EXPECTED_DATA_TYPES,
    },
    preprocessing_contract=PREPROCESSING_CONTRACT,
    prohibited_predictors=(
        *IDENTIFIER_COLUMNS,
        TARGET_COLUMN,
        "target-derived metrics",
    ),
)

split_manifest = build_split_manifest(
    dataset_slug=DATASET_SLUG,
    policy=SPLIT_POLICY,
    partitions=partitions,
    validation=partition_validation,
    partition_paths=PARTITION_RELATIVE_PATHS,
    partition_sha256=PARTITION_SHA256,
)

quality_evidence = build_quality_evidence(
    dataset_slug=DATASET_SLUG,
    raw_report=raw_validation,
    prepared_report=prepared_validation,
    partition_report=partition_validation,
    preparation=prepared_result,
    fingerprints={
        "source_sha256_before": SOURCE_SHA256_BEFORE,
        "source_sha256_after": SOURCE_SHA256_AFTER_PREPARATION,
        "raw_logical_fingerprint_before": RAW_LOGICAL_FINGERPRINT_BEFORE,
        "raw_logical_fingerprint_after": RAW_LOGICAL_FINGERPRINT_AFTER,
        "prepared_sha256": PREPARED_SHA256,
        "partition_sha256": PARTITION_SHA256,
    },
    readiness=READINESS_STATES,
    preservation_checks={
        "raw_dataframe_unchanged": True,
        "source_file_unchanged": True,
        "rows_removed": 0,
        "customer_ids_changed": 0,
        "valid_categories_changed": 0,
        "column_order_preserved": True,
        "prepared_row_order_preserved": True,
        "duplicate_feature_profiles_removed": 0,
        "generic_outlier_treatment_applied": False,
    },
)

write_result = write_preparation_artifacts(
    project_root=PROJECT.root,
    csv_artifacts={
        PREPARED_RELATIVE_PATH: prepared_df,
        PARTITION_RELATIVE_PATHS["train"]: train_df,
        PARTITION_RELATIVE_PATHS["validation"]: validation_df,
        PARTITION_RELATIVE_PATHS["test"]: test_df,
    },
    json_artifacts={
        MANIFEST_RELATIVE_PATHS["preparation_manifest"]: preparation_manifest,
        MANIFEST_RELATIVE_PATHS["feature_manifest"]: feature_manifest,
        MANIFEST_RELATIVE_PATHS["split_manifest"]: split_manifest,
        MANIFEST_RELATIVE_PATHS["quality_evidence"]: quality_evidence,
    },
    overwrite=False,
)

display(
    pd.DataFrame(
        [
            {"path": path, "status": status, "sha256": dict(write_result.sha256)[path]}
            for path, status in write_result.statuses
        ]
    )
)


,path,status,sha256
0,artifacts/preparation/telco-customer-churn/fea...,created,d0a57ff12aba2c462b0472e5f10d9d65634d1fb2ccc1fc...
1,artifacts/preparation/telco-customer-churn/pre...,created,0da44b7de064467a1705bdfbf1e72d2444a9ac22cbc016...
2,artifacts/preparation/telco-customer-churn/qua...,created,14baf9741251abadf32127844b815497fea4b80cd4d6c4...
3,artifacts/preparation/telco-customer-churn/spl...,created,bac2ed1eca1c90da79531ef579dfdbb551e3d16521704e...
4,data/processed/telco-customer-churn/prepared.csv,created,2b6eb0af34b82c8903bc06a8011cba1cc15ccfbda4e67d...
5,data/processed/telco-customer-churn/splits/str...,created,f6c99adfe3854643ca70ec4910c58645ba39fe1ec75b54...
6,data/processed/telco-customer-churn/splits/str...,created,58c173ac8fd138cbc7a38315f1f8cfed6896c1fe2b4776...
7,data/processed/telco-customer-churn/splits/str...,created,7386d73fc22881e970ec143703ad4315a1286643319636...


## 14. Preparation Findings

The prepared projection changes only `TotalCharges`, preserves all rows, identifiers, readable target values, valid category states, feature-profile repetition, source order, and source bytes. Snapshot splitting is ready for educational comparison, while temporal, inference-availability, and operational validity remain unresolved or unconfirmed.


In [14]:
PREPARATION_FINDINGS = {
    "authorized_value_corrections": {
        "TotalCharges": TOTAL_CHARGES_MATERIALIZED,
    },
    "rows_removed": 0,
    "identifiers_changed": 0,
    "categories_collapsed": 0,
    "feature_profiles_deduplicated": 0,
    "generic_outlier_treatment_applied": False,
    "learned_preprocessing_fitted": False,
    **READINESS_STATES,
}

display(pd.DataFrame([PREPARATION_FINDINGS]))


,authorized_value_corrections,rows_removed,identifiers_changed,categories_collapsed,feature_profiles_deduplicated,generic_outlier_treatment_applied,learned_preprocessing_fitted,deterministic_preparation_ready,prepared_dataset_materialized,benchmark_split_ready,benchmark_partitions_materialized,educational_model_selection_ready,operational_modeling_ready,temporal_contract_status,feature_inference_availability,operational_validity
0,{'TotalCharges': 11},0,0,0,0,False,False,True,True,True,True,True,False,unresolved,unconfirmed,unconfirmed


## 15. Model-Selection Handoff

Notebook 03 must load and validate these persisted artifacts rather than silently generating a different split. It may construct fold-local preprocessing pipelines from the feature manifest, but the fixed test partition must remain isolated until final evaluation.


In [15]:
handoff = load_and_validate_preparation_handoff(
    project_root=PROJECT.root,
    preparation_manifest_path=MANIFEST_RELATIVE_PATHS["preparation_manifest"],
    feature_manifest_path=MANIFEST_RELATIVE_PATHS["feature_manifest"],
    split_manifest_path=MANIFEST_RELATIVE_PATHS["split_manifest"],
    quality_evidence_path=MANIFEST_RELATIVE_PATHS["quality_evidence"],
)

if len(handoff.prepared) != EXPECTED_SOURCE_ROWS:
    raise RuntimeError("Validated handoff prepared row count is incorrect.")
if tuple(handoff.prepared.columns) != COLUMN_ORDER:
    raise RuntimeError("Validated handoff column order is incorrect.")
if handoff.manifests["split_manifest"]["operational_modeling_ready"] is not False:
    raise RuntimeError("Operational modeling readiness was incorrectly enabled.")

source_sha_final = fingerprint_file(DATASET_FILE)
if source_sha_final != SOURCE_SHA256_BEFORE:
    raise RuntimeError("Source CSV changed after artifact materialization.")
pd.testing.assert_frame_equal(raw_df, RAW_SNAPSHOT)

ignore_candidates = [
    PREPARED_RELATIVE_PATH,
    PARTITION_RELATIVE_PATHS["train"],
    MANIFEST_RELATIVE_PATHS["preparation_manifest"],
]
ignore_results = []
if shutil.which("git"):
    for relative in ignore_candidates:
        completed = subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT.root),
                "check-ignore",
                "--no-index",
                relative.as_posix(),
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        ignore_results.append(
            {
                "path": relative.as_posix(),
                "ignored": completed.returncode == 0,
            }
        )
else:
    ignore_results = [
        {"path": relative.as_posix(), "ignored": "git unavailable"}
        for relative in ignore_candidates
    ]

handoff_summary = pd.DataFrame(
    [
        {
            "prepared_path": PREPARED_RELATIVE_PATH.as_posix(),
            "prepared_sha256": PREPARED_SHA256,
            "train_rows": len(handoff.train),
            "validation_rows": len(handoff.validation),
            "test_rows": len(handoff.test),
            "educational_model_selection_ready": True,
            "operational_modeling_ready": False,
        }
    ]
)
display(handoff_summary)
display(pd.DataFrame(ignore_results))
print("Handoff validated without creating another split.")


,prepared_path,prepared_sha256,train_rows,validation_rows,test_rows,educational_model_selection_ready,operational_modeling_ready
0,data/processed/telco-customer-churn/prepared.csv,2b6eb0af34b82c8903bc06a8011cba1cc15ccfbda4e67d...,4930,1056,1057,True,False


,path,ignored
0,data/processed/telco-customer-churn/prepared.csv,True
1,data/processed/telco-customer-churn/splits/str...,True
2,artifacts/preparation/telco-customer-churn/pre...,True


Handoff validated without creating another split.
